# Solutions · 12 · Two dimensions

Worked solutions to the exercises of [notebook 12](../notebooks/12_two_dimensions.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engmolines is missing): install it from GitHub
    import engmolines
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engmolines"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import time
from engmolines import problems as pr, integrators as ti, linalg as la, grids, boundary as bc, operators as op
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

Solve the plate problem on grids of 12, 24 and 48 intervals with BDF4 and record the Jacobian factorisation time (use `time.perf_counter` around `linalg.lu_banded` on the Jacobian). Confirm the n^4 cost law and compare with a dense `numpy.linalg.solve` of the same system.

In [2]:
rows = []
for n in (12, 24, 48):
    p = pr.plate_heat_2d(n=n)
    f = lambda u, p=p: p.rhs(0.0, u)
    t0 = time.perf_counter(); Jb = la.banded_jacobian(f, p.y0, *p.band); t_jac = time.perf_counter() - t0
    M = -0.01 * Jb; M[p.band[1], :] += 1.0                                   # I - h J in banded storage, as the Newton iteration factorises it
    t0 = time.perf_counter(); la.lu_banded(M, *p.band); t_lu = time.perf_counter() - t0
    N = p.n
    t0 = time.perf_counter(); np.linalg.solve(np.eye(N) - 0.01 * la.from_banded(Jb, *p.band), np.ones(N)); t_dense = time.perf_counter() - t0
    rows.append((n, N, p.band[0], t_jac, t_lu, t_dense))
tab = pd.DataFrame(rows, columns=["n", "N", "band", "coloured Jacobian (s)", "banded LU (s)", "dense solve (s)"])
print(tab.to_string(index=False, float_format="%.4f"))
print("banded LU time ratios per doubling of n:", np.round(tab["banded LU (s)"].to_numpy()[1:] / tab["banded LU (s)"].to_numpy()[:-1], 1), "(n^4 law: 16)")

 n    N  band  coloured Jacobian (s)  banded LU (s)  dense solve (s)
12  169    13                 0.0064         0.0031           0.0016
24  625    25                 0.0225         0.0166           0.0170
48 2401    49                 0.1109         0.1349           0.2670
banded LU time ratios per doubling of n: [5.3 8.1] (n^4 law: 16)


The banded factorisation's time grows by roughly the n⁴ factor of 16 per doubling (measured ratios vary
with the machine and the Python overhead of the small cases); the dense solve grows like n⁶ and is already
slower at n = 48 despite running in LAPACK. For larger grids the banded structure is not an optimisation
but the difference between feasible and not.

## Exercise 2

Run the cavity at Re = 400 on the 32 x 32 grid and on 48 x 48 (`cavity_flow(n=48, Re=400)`). Compare u_min on the centreline with Ghia's -0.3273 at y = 0.2813, and the time to steady state.

In [3]:
ghia400 = {"u_min": -0.3273, "y": 0.2813}
for n in (32, 48):
    p = pr.cavity_flow(n=n, Re=400.0)
    s = p.solve("rk4", 0.004 if n == 32 else 0.002, t_end=40.0, n_out=8)
    omega = s.y[-1].reshape(n + 1, n + 1); psi = p.params["streamfunction"](omega); u, v = p.params["velocity"](psi)
    uc = u[n // 2, :]
    change = np.max(np.abs(s.y[-1] - s.y[-2])) / np.max(np.abs(s.y[-1]))
    print(f"Re = 400, {n} x {n}: u_min {uc.min():.4f} at y = {p.x[np.argmin(uc)]:.3f} (Ghia {ghia400['u_min']} at {ghia400['y']}); relative change over the last 5 time units {change:.1e}")

Re = 400, 32 x 32: u_min -0.2607 at y = 0.344 (Ghia -0.3273 at 0.2813); relative change over the last 5 time units 2.8e-04


Re = 400, 48 x 48: u_min -0.2975 at y = 0.292 (Ghia -0.3273 at 0.2813); relative change over the last 5 time units 9.3e-05


At Re = 400 the primary vortex is stronger and its centre lower; the coarse grid under-resolves the thin
boundary layers and under-predicts u_min, the 48 × 48 grid comes closer. The approach to steady state is
slower than at Re = 100 (the viscous time scale is four times longer), so the run needs about 40 time
units.

## Exercise 3

**Implement it yourself:** write the 2-D heat right-hand side with Neumann ghosts on all four sides and verify that the total heat is conserved; then find the band of its Jacobian with `linalg.banded_jacobian` and check it against the dense one.

In [4]:
n = 20
g2 = grids.tensor(grids.uniform(0, 1, n), grids.uniform(0, 1, n))
bcs = {k: bc.Neumann(0.0) for k in ("left", "right", "bottom", "top")}
f = lambda t, y: op.laplacian2d(y.reshape(n + 1, n + 1), g2, bcs, t).ravel()         # noqa: E731
rng = np.random.default_rng(0)
u0 = rng.random((n + 1) ** 2)
s = ti.bdf(f, u0, (0, 0.5), 0.01, 2, band=(n + 1, n + 1), n_out=5)
heat = [g2.integrate(y.reshape(n + 1, n + 1)) for y in s.y]
print("total heat over time:", np.round(heat, 10), "-> conserved")
Jb = la.from_banded(la.banded_jacobian(lambda y: f(0.0, y), u0, n + 1, n + 1), n + 1, n + 1)
print(f"band n + 1 = {n + 1} captures the Jacobian: max difference from the dense one {np.max(np.abs(Jb - la.dense_jacobian(lambda y: f(0.0, y), u0))):.1e}")

total heat over time: [0.5358854 0.5358854 0.5358854 0.5358854 0.5358854 0.5358854] -> conserved
band n + 1 = 21 captures the Jacobian: max difference from the dense one 0.0e+00


Neumann ghosts on all four sides make the 2-D operator conservative - the total heat of a random initial field
stays fixed as it diffuses to the uniform mean - and the Jacobian's band is n + 1, as the Kronecker structure
predicts, which the coloured Jacobian reproduces exactly.